## GPT Model

In [ ]:
CHOOSE_MODEL="gpt2-small (124M)"

BASE_CONFIG={
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs={
  "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
  "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
  "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
  "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len=in_idx.shape
    tok_embeds=self.tok_emb(in_idx)
    pos_embeds=self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x=tok_embeds+pos_embeds
    x=self.drop_emb(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits
  
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      dropout=cfg["drop_rate"],
      num_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut=x
    # Every row in the input has 0 mean and 1 variance
    x=self.norm1(x)
    # We get the context vector of [batch_size, num_tokens, emb_dim]
    x=self.att(x)
    # Dropout layer to improve efficiency
    x=self.drop_shortcut(x)
    # Creating shortcut connection
    x=x+shortcut

    # Shortcut for feed forward block
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    return x
  
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs
  
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift
  
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu
  
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)

def load_weights_into_gpt(gpt, params):
  gpt.pos_emb.weight=assign(gpt.pos_emb.weight, params["wpe"])
  gpt.tok_emb.weight=assign(gpt.tok_emb.weight, params["wte"])

  for b in range(len(params["blocks"])):
    q_w, k_w, v_w=np.split(
      params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.weight=assign(
      gpt.trf_blocks[b].att.w_query.weight, q_w.T
    )
    gpt.trf_blocks[b].att.w_key.weight=assign(
      gpt.trf_blocks[b].att.w_key.weight, k_w.T
    )
    gpt.trf_blocks[b].att.w_value.weight=assign(
      gpt.trf_blocks[b].att.w_value.weight, v_w.T
    )

    q_b, k_b, v_b=np.split(
      params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.bias=assign(
      gpt.trf_blocks[b].att.w_query.bias, q_b
    )
    gpt.trf_blocks[b].att.w_key.bias=assign(
      gpt.trf_blocks[b].att.w_key.bias, k_b
    )
    gpt.trf_blocks[b].att.w_value.bias=assign(
      gpt.trf_blocks[b].att.w_value.bias, v_b
    )

    gpt.trf_blocks[b].att.out_proj.weight=assign(
      gpt.trf_blocks[b].att.out_proj.weight,
      params["blocks"][b]["attn"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].att.out_proj.bias=assign(
      gpt.trf_blocks[b].att.out_proj.bias,
      params["blocks"][b]["attn"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].ff.layers[0].weight=assign(
      gpt.trf_blocks[b].ff.layers[0].weight,
      params["blocks"][b]["mlp"]["c_fc"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[0].bias=assign(
      gpt.trf_blocks[b].ff.layers[0].bias,
      params["blocks"][b]["mlp"]["c_fc"]["b"]
    )
    gpt.trf_blocks[b].ff.layers[2].weight=assign(
      gpt.trf_blocks[b].ff.layers[2].weight,
      params["blocks"][b]["mlp"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[2].bias=assign(
      gpt.trf_blocks[b].ff.layers[2].bias,
      params["blocks"][b]["mlp"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].norm1.scale=assign(
      gpt.trf_blocks[b].norm1.scale,
      params["blocks"][b]["ln_1"]["g"]
    )
    gpt.trf_blocks[b].norm1.shift=assign(
      gpt.trf_blocks[b].norm1.shift,
      params["blocks"][b]["ln_1"]["b"]
    )
    gpt.trf_blocks[b].norm2.scale=assign(
      gpt.trf_blocks[b].norm2.scale,
      params["blocks"][b]["ln_2"]["g"]
    )
    gpt.trf_blocks[b].norm2.shift=assign(
      gpt.trf_blocks[b].norm2.shift,
      params["blocks"][b]["ln_2"]["b"]
    )

  gpt.final_norm.scale=assign(gpt.final_norm.scale, params["g"])
  gpt.final_norm.shift=assign(gpt.final_norm.shift, params["b"])
  gpt.out_head.weight=assign(gpt.out_head.weight, params["wte"])

def assign(left, right):
  if left.shape!=right.shape:
    raise ValueError(f'Shape mismatch. Left Shape: {left.shape}, Right Shape: {right.shape}')
  return torch.nn.Parameter(torch.tensor(right, dtype=left.dtype, device=left.device))

def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx is (batch_size, num_tokens) array of indices in current context
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    '''For example, 
    Case 1: If LLM supports only 5 tokens and context size is
    10, then only the last 5 tokens are used as context.
    Case 2: If LLM supports 8 tokens and context size is 5, then
    only the last 5 tokens are used as context.'''
    idx_cond=idx[:, -context_size:]

    # Get output tensors - (batch_size, num_tokens, vocab_size)
    with torch.no_grad():
      logits=model(idx_cond)

    # Extract last vector
    logits=logits[:, -1, :]

    # Apply softmax to get probabilities - (batch_size, vocab_size)
    probs=torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next=torch.argmax(probs, dim=-1, keepdim=True)
    # (batch_size, 1)

    # Append sampled index to the running sequence
    idx=torch.cat((idx, idx_next), dim=1)
    # (batch_size, num_tokens+1)

  return idx

def text_to_token_ids(text, tokenizer):
  encoded_text=tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded_text).unsqueeze(0)
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat=token_ids.squeeze(0)
  decoded_text=tokenizer.decode(flat.tolist())
  return decoded_text

In [ ]:
model_size=CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

from gpt_download import download_and_load_gpt2

settings, params=download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model=GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()